# TIM Example 1 — Time base and update events

**Target:** STM32F103C8T6 · TIM4  
**Focus:** Build periodic timing from the timer clock, prescaler, and auto-reload value.

## Feature classification

| Variant | HAL start | CPU involvement | Typical purpose |
|---|---|---|---|
| Free-running time base | `HAL_TIM_Base_Start` | Read CNT / poll flags | Timestamping, periodic elapsed-time checks |
| Update interrupt | `HAL_TIM_Base_Start_IT` | ISR each update | Periodic control/sample trigger |
| Update DMA | `HAL_TIM_Base_Start_DMA` | DMA on update (target/buffer dependent) | Repeated hardware-paced transfer |
| Counter mode | Up / down / center-aligned where supported | Hardware counts | Time base or PWM carrier |
| Clock source | Internal, external clock mode, trigger/slave modes | Hardware-selected | Synchronization and event counting |

STM32F103C8T6 has TIM1–TIM4, all with 16-bit counter/ARR in this device; TIM1 is advanced-control, TIM2–TIM4 are general-purpose. Do not copy TIM6/TIM7 examples from other STM32 families.

## Start with the repository's TIM4 settings

The project `STM32F103C8T_Basic_Timer` uses TIM4, PSC=7199, ARR=65535. With a 72 MHz timer clock, the counter is 10 kHz (one count is 0.1 ms) and update occurs every 6.5536 s. Its `delay_ms(ms)` waits for `ms` counts, so the current implementation is about 10 times too short: `delay_ms(1000)` is about 100 ms. Correct the helper by converting milliseconds to counts (10 counts/ms) and checking the counter range.

For a separate 1 kHz update exercise, keep PSC=7199 and set ARR=9: 72 MHz / (7200 × 10) = 1 kHz. CubeMX's Prescaler/Period fields are register values (the HAL fields are one less than the desired division/count values).

Configure TIM4 as an up-counter, prescaler 7199, period 65535, clock division 1, internal clock to reproduce the project. Its interrupt start is commented out in the current source; begin with polling, then enable the matching timer IRQ and switch to interrupt mode.

```c
// Polling variant: start once, then poll for updates.
HAL_TIM_Base_Start(&htim4);
while (1) {
    if (__HAL_TIM_GET_FLAG(&htim4, TIM_FLAG_UPDATE) != RESET) {
        __HAL_TIM_CLEAR_FLAG(&htim4, TIM_FLAG_UPDATE);
        // At least one timer update has elapsed; polling can coalesce missed updates.
    }
}
```


## Update-interrupt variant

Start with `HAL_TIM_Base_Start_IT(&htim4)`. HAL routes the IRQ to the period callback when the interrupt handler calls `HAL_TIM_IRQHandler(&htim4)`. Keep the callback short; it runs at the update frequency.

```c
static volatile uint32_t update_count;

void HAL_TIM_PeriodElapsedCallback(TIM_HandleTypeDef *htim)
{
    if (htim->Instance == TIM4) {
        update_count++;
    }
}
```


## Timing checklist

- Determine `f_TIM` from APB prescaler and timer-clock doubling, not just HCLK.
- Use `(PSC + 1)` and `(ARR + 1)` in calculations.
- A 16-bit ARR tops out at 65535; use a larger PSC or cascade/synchronize timers for longer periods.
- Interrupt latency and callback duration limit usable update rates; do not assume a 1 kHz HAL callback is exact wall-clock time without measurement.